# ITAI 1371 — Midterm: Stroke Dataset Preparation
**Group:** BetaThetas  
**Purpose:** Explore and prepare the training portion of a stroke dataset for machine learning in the final project.

**Important:** The original `dataset.csv` is never changed. This notebook first splits the data into 70% training and 30% testing. All exploration, cleaning, encoding, scaling, and class balancing happen **only on the training data**. The testing data is saved unchanged for later use.

**Dataset source:** Confirm the exact Kaggle URL for the file your group downloaded before submitting. A similar 43,400-row dataset is listed as *Cerebral Stroke Prediction — Imbalanced Dataset*.

**Future ML question:** Can a classification model predict the `stroke` value (0 or 1) from the other available columns? This would be an educational exercise, not a medical diagnosis tool.

## 1. Load the original file

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Upload the CSV without editing it.
uploaded = files.upload()
df = pd.read_csv('dataset.csv')
print('Original dataset shape:', df.shape)
print(df.head())

## 2. Split first: training 70%, testing 30%
The testing portion remains untouched. All following analysis uses `train_df`.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.30, random_state=42)
train_df = train_df.copy()
test_df = test_df.copy()
print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))

# Save the unchanged test rows so they can be used in the final project.
test_df.to_csv('stroke_test_untouched.csv', index=False)

## 3. Explore training data before cleaning

In [ ]:
print('Training data information:')
train_df.info()
print('\nMissing values:')
print(train_df.isnull().sum())
print('\nStroke class counts:')
print(train_df['stroke'].value_counts())
print('\nSample before processing:')
print(train_df.head())

In [ ]:
plt.figure(figsize=(6,4))
train_df['stroke'].value_counts().sort_index().plot(kind='bar')
plt.title('Training Data: Stroke Classes Before Balancing')
plt.xlabel('Stroke (0 = No, 1 = Yes)')
plt.ylabel('Number of Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.hist(train_df['avg_glucose_level'].dropna(), bins=35)
plt.title('Training Data: Glucose Before Normalization')
plt.xlabel('Average Glucose Level')
plt.ylabel('Number of Records')
plt.tight_layout()
plt.show()

## 4. Fill missing values using training data only
The median is used for missing BMI values. Missing smoking status is labeled `Unknown` instead of assuming a person smoked or never smoked.

In [ ]:
clean = train_df.copy()
clean['bmi'] = clean['bmi'].fillna(clean['bmi'].median())
clean['smoking_status'] = clean['smoking_status'].fillna('Unknown')
print('Missing values after cleaning:')
print(clean.isnull().sum())
print(clean.head())

## 5. Feature engineering and normalization
Create an age-group feature from age. Use a logarithm to reduce the long right tail of glucose values. This is a basic example of changing a distribution, not a guarantee of better future model performance.

In [ ]:
clean['age_group'] = pd.cut(clean['age'], bins=[-1, 18, 40, 60, 120],
                            labels=['0-18', '19-40', '41-60', '61+'])
clean['glucose_log'] = np.log1p(clean['avg_glucose_level'])
print(clean[['age', 'age_group', 'avg_glucose_level', 'glucose_log']].head())

fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].hist(clean['avg_glucose_level'], bins=35)
axes[0].set_title('Glucose Before Normalization')
axes[0].set_xlabel('Average Glucose Level')
axes[0].set_ylabel('Number of Records')
axes[1].hist(clean['glucose_log'], bins=35)
axes[1].set_title('Glucose After Log Transformation')
axes[1].set_xlabel('Log of Glucose Level')
axes[1].set_ylabel('Number of Records')
plt.tight_layout()
plt.show()

## 6. Encode categorical columns
Map Yes/No to 1/0. Use `pd.get_dummies()` for other categories. The unique ID is removed from the training features because it identifies records rather than describing health characteristics.

In [ ]:
clean['ever_married'] = clean['ever_married'].map({'No': 0, 'Yes': 1})
clean = clean.drop(columns=['id', 'avg_glucose_level'])
clean = pd.get_dummies(clean, columns=['gender', 'work_type', 'Residence_type',
                                      'smoking_status', 'age_group'], dtype=int)
print('Sample after encoding:')
print(clean.head())
print('Columns:', clean.columns.tolist())

## 7. Scale numerical features
StandardScaler uses the mean and standard deviation from the **training data only**. The target column `stroke` is not scaled.

In [ ]:
scale_columns = ['age', 'bmi', 'glucose_log']
print('Before scaling:')
print(clean[scale_columns].head())
scaler = StandardScaler()
clean[scale_columns] = scaler.fit_transform(clean[scale_columns])
print('After scaling:')
print(clean[scale_columns].head())

## 8. Balance the training classes
Stroke examples are much less common. For this class-balancing demonstration, we randomly repeat rows from the smaller class **only within the training data**. Repeated rows should not be treated as new independent patients. Class balancing is for the training set, not the test set.

In [ ]:
class_0 = clean[clean['stroke'] == 0]
class_1 = clean[clean['stroke'] == 1]
print('Before balancing:', clean['stroke'].value_counts().to_dict())

# Repeat examples from the smaller class to match the larger class.
class_1_balanced = class_1.sample(n=len(class_0), replace=True, random_state=42)
clean_balanced = pd.concat([class_0, class_1_balanced], ignore_index=True)
clean_balanced = clean_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print('After balancing:', clean_balanced['stroke'].value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10,4))
clean['stroke'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Before Class Balancing')
axes[0].set_xlabel('Stroke Class')
axes[0].set_ylabel('Number of Records')
axes[0].tick_params(axis='x', rotation=0)
clean_balanced['stroke'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('After Class Balancing')
axes[1].set_xlabel('Stroke Class')
axes[1].set_ylabel('Number of Records')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 9. Compare samples and save results
The files exported below are created by Python code. **Do not edit CSV files manually.** The untouched testing file is saved separately, and the cleaned training file is what will be used for training later.

In [ ]:
print('Original training sample:')
print(train_df.head(3))
print('\nClean training sample before balancing:')
print(clean.head(3))
print('\nFinal balanced training sample:')
print(clean_balanced.head(3))
print('\nFinal clean training shape:', clean_balanced.shape)
print('Null values in cleaned training:', clean_balanced.isnull().sum().sum())
print('Untouched testing shape:', test_df.shape)

clean_balanced.to_csv('stroke_train_clean_balanced.csv', index=False)
clean.to_csv('stroke_train_clean_unbalanced.csv', index=False)
print('Saved stroke_train_clean_balanced.csv')
print('Saved stroke_train_clean_unbalanced.csv')
print('Saved stroke_test_untouched.csv')

## 10. What this dataset will be used for in the final project
In the final project, compare classification models that predict `stroke` (0 or 1) from age, BMI, glucose, and other available columns. Because the stroke class is uncommon in the original data, accuracy alone could be misleading. Precision, recall, F1-score, and a confusion matrix would help show whether the models find the minority class. The held-out test data should be processed later using choices learned from the training data—not inspected or fitted during this midterm.